# Source E: Medi-Cal FFS provider list

This notebook consolidates the statewide Medi-Cal FFS provider list to San Bernardino and Riverside counties, retains provider and geographic attributes, cleans the data, and saves a deduplicated Inland Empire extract.

Run the code cells from top to bottom with Shift+Enter.

## 1. Find the data and import pandas

In [ ]:
from pathlib import Path
import pandas as pd

relative_partition_dir = Path(
    "data/provider-side/E-mediCal-fee-for-service-enrolled-providers/"
    "zipped_bc_too_big_for_github"
)

repo_root = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / relative_partition_dir).is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the Source E data from the notebook's current folder: "
        f"{Path.cwd()}. Start Jupyter with the repository folder as its root."
    )

partition_dir = repo_root / relative_partition_dir
print("Source E folder:", partition_dir)


## 2. Load the two CSV parts

In [ ]:
csv_files = [
    partition_dir / "medi-cal-ffs-provider-listing_10_26_26_PART 1 first 200k line.csv",
    partition_dir / "medi-cal-ffs-provider-listing_10_26_26_PART 2 next 180k line.csv",
]

missing_files = [path for path in csv_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Missing expected Source E CSV file(s): "
        + ", ".join(str(path) for path in missing_files)
    )

providers_df = pd.concat(
    [
        pd.read_csv(
            path,
            encoding="cp1252",
            dtype=str,  # Preserve identifiers and leading zeros.
            low_memory=False,
            keep_default_na=False,
            na_values=[""],
        )
        for path in csv_files
    ],
    ignore_index=True,
)

required_columns = {"County", "CountyName", "State", "NPI", "Provider_Number"}
missing_columns = sorted(required_columns - set(providers_df.columns))
if missing_columns:
    raise ValueError(f"Source CSV is missing required columns: {missing_columns}")

print("Loaded both CSV parts into providers_df.")
print("Rows loaded:", len(providers_df))


## 3. Clean, subset to San Bernardino + Riverside, and deduplicate

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).replace("\u00a0", " ").split())

providers_df = providers_df.copy()

# Normalize text values and whitespace consistently.
for col in providers_df.columns:
    providers_df[col] = providers_df[col].map(normalize_text)

providers_df["County_norm"] = providers_df["County"].str.upper()
providers_df["CountyName_norm"] = providers_df["CountyName"].str.upper()
providers_df["State_norm"] = providers_df["State"].str.upper()

county_targets = {"SAN BERNARDINO", "RIVERSIDE"}
county_name_targets = {"SAN BERNARDINO COUNTY", "RIVERSIDE COUNTY"}
is_target_county = (
    providers_df["County_norm"].isin(county_targets)
    | providers_df["CountyName_norm"].isin(county_name_targets)
)
inland_empire_df = providers_df[is_target_county & providers_df["State_norm"].eq("CA")].copy()

print("Rows in statewide dataset:", len(providers_df))
print("Rows after California county filter:", len(inland_empire_df))
print("County values after filter:")
print(inland_empire_df["County"].value_counts(dropna=False))

# Use NPI when present, otherwise Provider_Number. Give records with neither ID
# a row-specific key so unrelated providers are not grouped together.
npi = inland_empire_df["NPI"].str.strip()
provider_number = inland_empire_df["Provider_Number"].str.strip()
inland_empire_df["provider_key"] = npi.mask(npi.eq(""), provider_number)
missing_provider_key = inland_empire_df["provider_key"].eq("")
inland_empire_df.loc[missing_provider_key, "provider_key"] = [
    f"UNKNOWN-{row_id}" for row_id in inland_empire_df.index[missing_provider_key]
]

# Build a normalized address key while preserving records with no usable address.
address_cols = ["Address", "Address2", "City", "State", "ZIP"]
address_parts = inland_empire_df[address_cols].apply(
    lambda column: column.str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
)
inland_empire_df["address_key"] = address_parts.agg(" | ".join, axis=1)
missing_address = address_parts.eq("").all(axis=1)
inland_empire_df.loc[missing_address, "address_key"] = [
    f"UNKNOWN-ADDRESS-{row_id}" for row_id in inland_empire_df.index[missing_address]
]

duplicate_mask = inland_empire_df.duplicated(
    subset=["provider_key", "address_key"], keep=False
)
print("Rows in duplicate provider/address groups:", int(duplicate_mask.sum()))
if duplicate_mask.any():
    print("Sample duplicate records for review:")
    display(
        inland_empire_df.loc[duplicate_mask, ["Provider_Number", "NPI", *address_cols]]
        .head(10)
    )

# Keep the first row for exact provider/address duplicates; review the sample above
# if source ordering or conflicting duplicate attributes are a concern.
rows_before_deduplication = len(inland_empire_df)
inland_empire_df = inland_empire_df.drop_duplicates(
    subset=["provider_key", "address_key"], keep="first"
).copy()
print("Rows removed by provider+address deduplication:", rows_before_deduplication - len(inland_empire_df))

keep_cols = [
    "Provider_Number",
    "NPI",
    "NPI_Type",
    "Owner_Number",
    "Service_Location_Number",
    "Legal_Name",
    "Enroll_Status_Eff_DT",
    "Provider_Taxonomy",
    "NEMT_NMT_Provider_Type",
    "ANC_Provider_Type",
    "FI_Provider_Type_CD",
    "FI_Provider_Type",
    "Provider_License",
    "FI_Provider_Specialty_CD",
    "FI_Provider_Specialty",
    "Out_of_State_Indicator",
    "In_Out_State",
    "Address_Attention",
    "Address",
    "Address2",
    "City",
    "State",
    "ZIP",
    "ZIP_4",
    "DHCS_County_CD",
    "FIPS_County_CD",
    "County",
    "CountyName",
    "Phone_Number",
    "Medicaid_Patients",
    "CHIP_Patients",
    "Telehealth_Services",
    "Provider_Website",
    "Threshold_Languages",
    "Other_Languages",
    "Acc_Exam_Room",
    "Acc_Exterior_Building",
    "Acc_Interior_Building",
    "Acc_Parking",
    "Acc_Restroom",
    "Acc_Medical_Equipment",
    "Acc_Patient_Areas",
    "Acc_Patient_Diagnostic",
    "Provider_Source",
    "Billing_Provider",
    "Latitude",
    "Longitude",
]
missing_output_columns = sorted(set(keep_cols) - set(inland_empire_df.columns))
if missing_output_columns:
    raise ValueError(f"Source CSV is missing output columns: {missing_output_columns}")

inland_empire_clean = inland_empire_df[keep_cols].copy().reset_index(drop=True)
inland_empire_clean["County"] = inland_empire_clean["County"].replace({"": "Unknown"})
inland_empire_clean["CountyName"] = inland_empire_clean["CountyName"].replace({"": "Unknown"})

print("Final row count:", len(inland_empire_clean))
print("Rows by county:")
print(inland_empire_clean["County"].value_counts(dropna=False))
print("Top provider types:")
print(inland_empire_clean["FI_Provider_Type"].value_counts(dropna=False).head(10))
print("Missing/blank key-field counts:")
print(inland_empire_clean[["Provider_Number", "NPI", "Address", "City", "ZIP"]].eq("").sum())

assert inland_empire_df["State_norm"].eq("CA").all(), "Unexpected non-California rows found."
assert (
    inland_empire_df["County_norm"].isin(county_targets)
    | inland_empire_df["CountyName_norm"].isin(county_name_targets)
).all(), "Unexpected county rows found."

output_path = partition_dir / "san_bernardino_riverside_ffs_providers_clean.csv"
inland_empire_clean.to_csv(output_path, index=False)
print("Saved cleaned file to:", output_path)


## 4. Preview the clean Inland Empire table

In [ ]:
inland_empire_clean.head()


## 5. Final integrated IEHP dashboard analysis

Run all the Source E cells above first. This optional step exports provider data and, when the matching files are present, integrates Sources A, H, B, D, F, IEHP directory matches, and financial scenario results. In Colab/Jupyter you can also use the interactive filter and rank controls. The FFS source CSVs must be accessible in the path configured in Section 1.

In [ ]:
# IEHP CEO CASE — ONE-CELL DATA PIPELINE
# Paste into ONE new code cell at bottom of Source E notebook, then Run.
# Requires: pandas, openpyxl. Optional buttons: ipywidgets.
# Source E alone works. Other lettered sources are used only when their files exist.
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

ROOT = Path(globals().get('repo_root', Path.cwd()))
OUT = ROOT / 'analysis_outputs' / 'IEHP_FINAL_DASHBOARD'
OUT.mkdir(parents=True, exist_ok=True)
# If automatic discovery misses a file, put its full path here:
FILES = {'A':None,'H':None,'F':None,'I':None,'D':None,'B_age':None,'B_language':None,'FIN':None}
COUNTIES = {'RIVERSIDE':'Riverside','SAN BERNARDINO':'San Bernardino'}
FIPS = {'06065':'Riverside','06071':'San Bernardino'}
log = []
def status(source, result):
    log.append({'source':source,'status':result}); print(source, '-', result)
def pick(key, patterns):
    if FILES[key]:
        p = Path(FILES[key]).expanduser()
        if not p.exists(): raise FileNotFoundError(p)
        return p
    for pattern in patterns:
        matches = sorted(ROOT.glob(pattern))
        if matches: return matches[0]
    return None
def csv(path):
    try: return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig', low_memory=False)
    except UnicodeDecodeError: return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='cp1252', low_memory=False)
def npi(x):
    if pd.isna(x): return ''
    s = str(x).strip()
    if re.fullmatch(r'\d{10}\.0+',s): s = s.split('.')[0]
    if re.fullmatch(r'[\d.]+[eE][+-]?\d+',s):
        try:
            from decimal import Decimal
            val = Decimal(s)
            s = str(int(val)) if val == val.to_integral_value() else ''
        except Exception: s = ''
    return s if re.fullmatch(r'\d{10}',s) else ''
def valid_npi(s):
    if not re.fullmatch(r'\d{10}',s): return False
    d = [int(x) for x in '80840'+s]
    total = sum((2*v//10+2*v%10) if j%2 else v for j,v in enumerate(reversed(d)))
    return total%10 == 0
def zip5(s): return s.fillna('').astype(str).str.extract(r'(\d{5})',expand=False).fillna('')
def numeric(s): return pd.to_numeric(s.astype(str).str.replace(',','',regex=False).replace({'':'', '*':'', '-':''}),errors='coerce')

# 1) SOURCE E: FFS providers, all local providers and specialties
if 'providers_df' in globals():
    raw = providers_df.copy()
else:
    parts = sorted(ROOT.glob('**/medi-cal-ffs-provider-listing_10_26_26_PART*.csv'))
    if len(parts)!=2: raise FileNotFoundError('Upload both Source E PART 1 and PART 2 CSVs, or run the existing Source E loading cells first.')
    raw = pd.concat([pd.read_csv(p,encoding='cp1252',low_memory=False,keep_default_na=False) for p in parts],ignore_index=True)
need = {'NPI','State','CountyName','ZIP','FI_Provider_Type','FI_Provider_Specialty','Legal_Name','Address','City'}
if need-set(raw.columns): raise ValueError('Source E missing columns: '+str(need-set(raw.columns)))
e = raw.copy()
for c in need: e[c] = e[c].fillna('').astype(str).str.strip()
e['npi'] = e.NPI.map(npi); e['npi_ok'] = e.npi.map(valid_npi)
e['county'] = e.CountyName.str.upper().str.replace(' COUNTY','',regex=False).str.strip().map(COUNTIES)
e['zip'] = zip5(e.ZIP)
e_all_npis = set(e.loc[e.npi_ok,'npi'])  # statewide reference, before county filter
e = e.loc[e.State.str.upper().eq('CA') & e.county.notna()].copy()
e.loc[~e.npi_ok | ~e.zip.str.fullmatch(r'\d{5}')].to_csv(OUT/'E_records_to_review.csv',index=False)
e = e.loc[e.npi_ok & e.zip.str.fullmatch(r'\d{5}')].copy()
e['type'] = e.FI_Provider_Type; e['specialty'] = e.FI_Provider_Specialty
e['name'] = e.Legal_Name; e['city'] = e.City; e['address'] = e.Address
def category(row):
    s = (row['type']+' '+row['specialty']).lower()
    if any(t in s for t in ['psychiatr','psycholog','behavior','mental health','counsel','social work','marriage and family']): return 'Mental health (candidate)'
    if any(t in s for t in ['dental','dentist','orthodont']): return 'Dental (E; use F for complete dental)'
    if any(t in s for t in ['family practice','general practice','internal medicine','pediatrics','primary care']): return 'Primary care (candidate)'
    if any(t in s for t in ['physician','surgeon','cardiolog','neurolog','dermatolog']): return 'Specialist/physician (candidate)'
    return 'Other/unspecified'
e['service'] = e.apply(category,axis=1)
e['site'] = e.npi+'|'+e.county+'|'+e.zip+'|'+e.address.str.upper().str.replace(r'\s+',' ',regex=True)
e = e.drop_duplicates(['site','type','specialty']).copy()
e[['npi','name','county','city','zip','address','type','specialty','service','site']].to_csv(OUT/'01_E_FFS_provider_list.csv',index=False)
supply = e.groupby(['county','zip','service'],as_index=False).agg(ffs_npis=('npi','nunique'),ffs_sites=('site','nunique'))
supply.to_csv(OUT/'02_E_FFS_supply_by_ZIP_specialty.csv',index=False)
status('E',f'READY: {e.npi.nunique():,} distinct FFS NPIs in RC/SBC (not verified appointment capacity)')

# 2) SOURCE I: IEHP directory extracted from the two official PDFs (optional)
ifile = pick('I',['IEHP_FINAL_4_CSVs/01_IEHP_providers_Riverside_San_Bernardino.csv',
 'IEHP_PDF_FFS_Export/07_UNIQUE_IEHP_PROVIDERS_DEDUPED_BY_NPI.csv','data/*IEHP*provider*.csv','**/IEHP_provider_directory_extracted.csv'])
if ifile:
    i = csv(ifile); column = next((c for c in ['npi','npi_as_printed','NPI'] if c in i),None)
    if not column: raise ValueError('IEHP CSV must contain an NPI column')
    i['npi'] = i[column].map(npi); i = i[i.npi.map(valid_npi)].copy()
    loc = set(e.npi); iehp = set(i.npi)
    i['ffs_statewide_match'] = i.npi.isin(e_all_npis)
    i['ffs_local_match'] = i.npi.isin(loc)
    i.to_csv(OUT/'03_I_IEHP_providers_FFS_flags.csv',index=False)
    pd.DataFrame([
        ('IEHP unique NPIs',len(iehp)),('Local FFS unique NPIs',len(loc)),
        ('Both IEHP and local FFS',len(iehp & loc)),
        ('Local FFS no match in supplied IEHP PDF',len(loc-iehp)),
        ('IEHP no match in local FFS',len(iehp-loc)),
        ('IEHP no match in FFS statewide',len(iehp-e_all_npis))],
        columns=['metric','count']).to_csv(OUT/'04_provider_overlap_numbers.csv',index=False)
    status('I',f'READY: {len(iehp & loc):,} IEHP NPIs match the local FFS roster')
else: status('I','MISSING: Upload extracted IEHP PDF CSV if provider overlap is needed')

# 3) SOURCES A + H: Medi-Cal member demand by ZIP (optional but REQUIRED for access risk)
afile = pick('A',['**/calendar-years-2020-to-the-most-recent-reportable-month.csv'])
hfile = pick('H',['**/ZIP-COUNTY_062026.xlsx','**/ZIP-COUNTY*.csv','**/ZIP-COUNTY*.xlsx'])
access = None
if afile and hfile:
    a = csv(afile)
    needed = {'Month of Eligibility','Zip Code','Sex','Total Eligibles'}
    if needed-set(a): raise ValueError('Source A columns missing: '+str(needed-set(a)))
    month = next((m for m in ['2026-09','2026-08'] if m in set(a['Month of Eligibility'])),None)
    if not month: raise ValueError('Use Source A for August or September 2026')
    a = a[a['Month of Eligibility'].eq(month)].copy()
    a['zip'] = zip5(a['Zip Code']); a = a[a.zip.str.fullmatch(r'\d{5}')]
    a['sex'] = a.Sex.str.lower().str.strip().map({'male':'male','m':'male','female':'female','f':'female','total':'total','t':'total'})
    a['value'] = numeric(a['Total Eligibles'])
    if a.duplicated(['zip','sex']).any(): raise ValueError('Source A duplicate ZIP/sex rows; investigate before using')
    a = a.pivot(index='zip',columns='sex',values='value')
    for c in ['male','female','total']:
        if c not in a: a[c] = float('nan')
    both = a.male+a.female
    conflicts = both.notna() & a.total.notna() & both.ne(a.total)
    a['members'] = both.combine_first(a.total)
    a.loc[conflicts,'members'] = float('nan')  # cannot assume which number is correct
    a = a.reset_index()[['zip','members']]
    h = pd.read_excel(hfile,dtype=str) if hfile.suffix.lower() in ['.xlsx','.xls'] else csv(hfile)
    h.columns = h.columns.astype(str).str.lower().str.strip()
    if not {'zip','geoid','res_ratio'}.issubset(h): raise ValueError('Source H needs zip, geoid, res_ratio')
    h['zip'] = zip5(h['zip']); h['geoid'] = h.geoid.astype(str).str.replace(r'\.0$','',regex=True).str.zfill(5)
    h['ratio'] = pd.to_numeric(h.res_ratio,errors='coerce')
    h = h[h.geoid.isin(FIPS) & h.ratio.gt(0) & h.ratio.le(1)].copy()
    if h.duplicated(['zip','geoid']).any(): raise ValueError('Source H duplicate ZIP/county rows; investigate')
    h['county'] = h.geoid.map(FIPS)
    demand = h[['county','zip','ratio']].merge(a,on='zip',how='left',validate='many_to_one')
    demand['medi_cal_members_est'] = demand.members*demand.ratio
    demand['month'] = month
    demand[['county','zip','ratio','medi_cal_members_est','month']].to_csv(OUT/'05_A_H_members_by_ZIP.csv',index=False)
    services = sorted(set(supply.service) - {'Dental (E; use F for complete dental)'})
    # Every ZIP with demand appears, including ZIPs with zero listed providers.
    index = demand[['county','zip']].merge(pd.DataFrame({'service':services}),how='cross')
    access = index.merge(supply,on=['county','zip','service'],how='left',validate='one_to_one')
    access['ffs_npis'] = access.ffs_npis.fillna(0).astype(int)
    access = access.merge(demand[['county','zip','medi_cal_members_est','month']],on=['county','zip'],how='left',validate='many_to_one')
    access['providers_per_1000'] = 1000*access.ffs_npis/access.medi_cal_members_est
    access['zero_ffs_listed'] = access.ffs_npis.eq(0)
    access['usable_for_ranking'] = access.medi_cal_members_est.gt(0)
    access = access.sort_values(['service','zero_ffs_listed','providers_per_1000','medi_cal_members_est'],ascending=[True,False,True,False],na_position='last')
    access['risk_rank'] = access[access.usable_for_ranking].groupby('service').cumcount()+1
    access.to_csv(OUT/'06_ACCESS_per_1000_and_ranking.csv',index=False)
    access[access.usable_for_ranking].groupby('service',group_keys=False).head(10).to_csv(OUT/'07_TOP10_access_gaps.csv',index=False)
    status('A+H',f'READY: {month}; access ratios use ALL Medi-Cal members, not confirmed transitioning IEHP members')
else: status('A+H','MISSING: upload A and H to calculate providers per 1,000 and rank access gaps')

# 4) OPTIONAL F, B, D, finance: add source context without inventing missing values
ffile = pick('F',['**/F_dental.csv','**/dental_ffs.csv'])
if ffile:
    f = csv(ffile); f.columns = f.columns.str.strip()
    id_col = next((c for c in ['NPI','npi','Provider NPI'] if c in f),None)
    zip_col = next((c for c in ['ZIP','Zip','Zip Code','ZIP Code'] if c in f),None)
    county_col = next((c for c in ['CountyName','County','county'] if c in f),None)
    if id_col and zip_col and county_col:
        f['npi'] = f[id_col].map(npi); f['zip'] = zip5(f[zip_col])
        f['county'] = (f[county_col].astype(str).str.upper()
                       .str.replace(' COUNTY','',regex=False).str.strip().map(COUNTIES))
        f = f[f.npi.map(valid_npi) & f.county.notna() & f.zip.str.fullmatch(r'\d{5}')].copy()
        dental = (f.groupby(['county','zip'],as_index=False).npi.nunique()
                  .rename(columns={'npi':'ffs_npis'}))
        dental['service'] = 'Dental (Source F)'
        dental.to_csv(OUT/'08_F_dental_by_ZIP.csv',index=False)
        if access is not None:
            index = demand[['county','zip']].drop_duplicates()
            dacc = index.merge(dental,on=['county','zip'],how='left')
            dacc['service'] = 'Dental (Source F)'
            dacc['ffs_npis'] = dacc.ffs_npis.fillna(0).astype(int)
            dacc = dacc.merge(demand[['county','zip','medi_cal_members_est','month']],on=['county','zip'],how='left')
            dacc['ffs_sites'] = pd.NA
            dacc['providers_per_1000'] = 1000*dacc.ffs_npis/dacc.medi_cal_members_est
            dacc['zero_ffs_listed'] = dacc.ffs_npis.eq(0)
            dacc['usable_for_ranking'] = dacc.medi_cal_members_est.gt(0)
            dacc = dacc.sort_values(['zero_ffs_listed','providers_per_1000','medi_cal_members_est'],ascending=[False,True,False],na_position='last')
            dacc['risk_rank'] = pd.NA
            dacc.loc[dacc.usable_for_ranking,'risk_rank'] = range(1,int(dacc.usable_for_ranking.sum())+1)
            access = pd.concat([access,dacc[access.columns]],ignore_index=True)
            access.to_csv(OUT/'06_ACCESS_per_1000_and_ranking.csv',index=False)
            access[access.usable_for_ranking].groupby('service',group_keys=False).head(10).to_csv(OUT/'07_TOP10_access_gaps.csv',index=False)
        status('F',f'READY: {f.npi.nunique():,} valid dental NPIs; check data date and availability')
    else:
        f.to_csv(OUT/'08_F_dental_raw_for_review.csv',index=False)
        status('F','REVIEW: dental columns do not match; raw Source F saved for manual classification')
else: status('F','MISSING: dental coverage requires Source F')
for key,patterns,name in [
    ('B_age',['analysis_outputs/source_b/latest_age_summary.csv'],'09_B_age_summary.csv'),
    ('B_language',['analysis_outputs/source_b/latest_language_summary.csv'],'10_B_language_summary.csv')]:
    p = pick(key,patterns)
    if p: csv(p).to_csv(OUT/name,index=False); status(key,'READY (county-level context)')
    else: status(key,'MISSING: run Source B notebook first')
dfile = pick('D',['**/open-data-portal-managed-care-enrollment-count-september-2026.csv'])
if dfile:
    d=csv(dfile)
    required={'Plan Name','Enrollment Month','County','Count of Enrollees'}
    if required.issubset(d):
        d=d[d['Enrollment Month'].eq('2026-09') & d.County.isin(['Riverside','San Bernardino'])
            & d['Plan Name'].str.startswith('Inland Emp',na=False)
            & ~d['Plan Name'].str.contains('CCI|CMC',na=False,regex=True)].copy()
        d['members'] = numeric(d['Count of Enrollees'])
        d.groupby('County',as_index=False).members.sum(min_count=1).to_csv(OUT/'11_D_IEHP_enrollment.csv',index=False)
        status('D','READY: plan enrollment, NOT transition member counts')
    else: status('D','REVIEW: Source D unexpected columns')
else: status('D','MISSING: optional enrollment baseline')
finance = pick('FIN',['analysis_outputs/financial/scenario_results.csv',
                     'IEHP_FINAL_VERIFIED_COLLECTION/09_Financial_scenarios_MODELED.csv'])
if finance:
    f=csv(finance)
    f.to_csv(OUT/'12_FINANCIAL_model_scenarios.csv',index=False)
    status('FIN','READY: scenario outputs are MODEL ESTIMATES, not confirmed IEHP losses')
else: status('FIN','MISSING: run the financial notebook or upload its scenario CSV')

# 5) Source C baseline, guide caveats, reproducibility / quality notes
pd.DataFrame([
    ['C','IEHP Medi-Cal fact-sheet members',1321983,'October 2026 fact sheet; count month unspecified'],
    ['A','Medi-Cal ZIP counts','','All eligible members, not necessarily transitioning members'],
    ['E','FFS provider listing','','FFS enrollment does NOT guarantee accepting new patients'],
    ['I','IEHP directory','','NPI overlap does not confirm same office or appointment availability'],
    ['F','Dental provider list','','Needed separately; do not imply dental supply from Source E'],
    ['G','HRSA shortage areas','','Optional regional context, not exact ZIP FFS capacity'],
    ['Case','Model estimates','','Use attributable assumptions; disclose AI-assisted code']
],columns=['source','metric','value','important_note']).to_csv(OUT/'13_SOURCE_NOTES.csv',index=False)
pd.DataFrame(log).to_csv(OUT/'00_READINESS.csv',index=False)
print('\nDashboard CSVs saved at:', OUT.resolve())
display(pd.DataFrame(log))

# 6) Interactive filtering/ranking buttons (Jupyter or Colab, NOT GitHub preview)
try:
    import ipywidgets as w
    from IPython.display import clear_output
    table = access if access is not None else supply
    county = w.Dropdown(options=['All','Riverside','San Bernardino'],description='County:')
    service = w.Dropdown(options=['All']+sorted(table.service.unique()),description='Service:')
    zip_input = w.Text(description='ZIP:',placeholder='optional')
    options = [('Fewest FFS providers','fewest'),('Most FFS providers','most')]
    if access is not None: options += [('Highest potential access risk','risk'),('Most members','members')]
    ranking = w.Dropdown(options=options,description='Rank:')
    top = w.Dropdown(options=[10,25,50,100,0],description='Top:',value=10)
    run_button=w.Button(description='Filter & Rank',button_style='primary')
    save_button=w.Button(description='Export displayed CSV')
    output=w.Output(); selected={'data':pd.DataFrame()}
    def run_filter(_=None):
        t=table.copy()
        if county.value!='All': t=t[t.county.eq(county.value)]
        if service.value!='All': t=t[t.service.eq(service.value)]
        if zip_input.value.strip(): t=t[t.zip.eq(zip_input.value.strip())]
        if ranking.value=='risk':
            t=t[t.usable_for_ranking].sort_values(['zero_ffs_listed','providers_per_1000','medi_cal_members_est'],ascending=[False,True,False])
        elif ranking.value=='members': t=t.sort_values('medi_cal_members_est',ascending=False)
        else: t=t.sort_values('ffs_npis',ascending=(ranking.value=='fewest'))
        if top.value: t=t.head(top.value)
        selected['data']=t
        with output:
            clear_output(wait=True); print('Results:',len(t),'| Supply ≠ accepting new patients'); display(t.head(100))
    def export_filter(_):
        if selected['data'].empty: run_filter()
        path=OUT/'14_CUSTOM_FILTERED_RESULTS.csv'
        selected['data'].to_csv(path,index=False)
        with output: print('Saved:',path)
    run_button.on_click(run_filter); save_button.on_click(export_filter)
    display(w.VBox([w.HTML('<b>IEHP Provider and Access Ranking</b>'),w.HBox([county,service]),
                     w.HBox([zip_input,ranking,top]),w.HBox([run_button,save_button]),output]))
    run_filter()
except ImportError:
    print('Interactive buttons optional: install with %pip install ipywidgets, then rerun this cell.')
